In [6]:
from pathlib import Path
from typing import List
from PIL import Image

In [10]:
def get_monitors() -> List[ROI]:
    """
    Return all available monitors (regions of interest).
    0 = Entire virtual desktop.
    1 = Primary screen.
    2 = Secondary screen, etc.
    """
    try:
        with mss.mss() as sct:
            return sct.monitors
    except Exception as e:
        raise ScreenCaptureError(f"Failed to enumerate monitors: {e}") from e


def grab_regions(screen_id: int, rois: List[ROI], img_format: str = "RGB") -> List[Image.Image]:
    """
    Low-level capture of one or more regions using mss.
    """
    if img_format not in ("RGB", "RGBA"):
        raise ScreenCaptureError(f"Invalid image format: {img_format}")

    try:
        with mss.mss() as sct:
            captured_images: List[Image.Image] = []
            for area in rois:
                raw_capture = sct.grab(area)
                if img_format == "RGB":
                    img = Image.frombytes("RGB", raw_capture.size, raw_capture.rgb)
                else:  # RGBA
                    img = Image.frombytes("RGBA", raw_capture.size, raw_capture.bgra)
                captured_images.append(img)
            return captured_images
    except Exception as e:
        raise ScreenCaptureError(f"Failed to grab regions: {e}") from e


def capture_screen_regions(
    screen_id: int,
    rois: Optional[List[ROI]] = None,
    img_format: str = "RGB"
) -> List[Image.Image]:
    """
    High-level API: Capture one or more regions from a specific screen.
    """
    monitors = get_monitors()
    if not (0 <= screen_id < len(monitors)):
        raise ScreenCaptureError(f"Invalid screen_id {screen_id}. Available: 0 to {len(monitors)-1}.")

    capture_areas: List[ROI] = rois or [monitors[screen_id]]
    return grab_regions(screen_id=screen_id, rois=capture_areas, img_format=img_format)


def get_half_width_roi(screen: ROI) -> ROI:
    """Return the left half of the given screen region."""
    return {
        "top": screen["top"],
        "left": screen["left"],
        "width": screen["width"] // 2,
        "height": screen["height"],
    }

def capture_half_screen(screen_id: int, img_format: str = "RGB") -> List[Image.Image]:
    """
    Capture the left half of a given screen and return it as a list of images.
    """
    monitors = get_monitors()
    if not (0 <= screen_id < len(monitors)):
        raise ScreenCaptureError(f"Invalid screen_id {screen_id}. Available: 0–{len(monitors)-1}.")

    half_roi = get_half_width_roi(monitors[screen_id])
    return capture_screen_regions(screen_id=screen_id, rois=[half_roi], img_format=img_format)


In [11]:
# Example 1: Capture entire primary screen
print("Capturing entire primary screen...")
full_screen_images = capture_screen_regions(screen_id=1)
if full_screen_images:
    full_screen_images[0].save("primary_screen_capture.png")
    print("Entire screen saved as 'primary_screen_capture.png'.")

print("-" * 20)

# Example 2: Capture entire secondary screen
print("Capturing entire secondary screen...")
full_screen_images = capture_screen_regions(screen_id=2)
if full_screen_images:
    full_screen_images[0].save("secondary_screen_capture.png")
    print("Entire screen saved as 'secondary_screen_capture.png'.")

print("-" * 20)

# Example 3: Capture entire secondary screen
print("Capturing entire secondary screen...")
full_screen_images = capture_half_screen(screen_id=2)
if full_screen_images:
    full_screen_images[0].save("half_width_screen_2_capture.png")
    print("Entire screen saved as 'half_width_screen_2_capture.png'.")

print("-" * 20)

# Example 3: Capture two specific regions from primary screen
print("Capturing two specific regions from primary screen...")
regions_to_capture: List[ROI] = [
    {"top": 100, "left": 100, "width": 400, "height": 200},
    {"top": 500, "left": 300, "width": 150, "height": 150}
]
roi_images = capture_screen_regions(screen_id=1, rois=regions_to_capture)

for i, img in enumerate(roi_images):
    file_path = f"roi_capture_{i+1}.png"
    img.save(file_path)
    print(f"Region {i+1} saved as '{file_path}'.")

print("-" * 20)

# Example 4: Capture region with RGBA format
print("Capturing region in RGBA format...")
rgba_region: List[ROI] = [{"top": 0, "left": 0, "width": 500, "height": 500}]
rgba_images = capture_screen_regions(screen_id=1, rois=rgba_region, img_format="RGBA")
if rgba_images:
    rgba_images[0].save("rgba_capture.png")
    print("Region saved as 'rgba_capture.png' (with alpha channel).")

Capturing entire primary screen...
Entire screen saved as 'primary_screen_capture.png'.
--------------------
Capturing entire secondary screen...
Entire screen saved as 'secondary_screen_capture.png'.
--------------------
Capturing entire secondary screen...
Entire screen saved as 'half_width_screen_2_capture.png'.
--------------------
Capturing two specific regions from primary screen...
Region 1 saved as 'roi_capture_1.png'.
Region 2 saved as 'roi_capture_2.png'.
--------------------
Capturing region in RGBA format...
Region saved as 'rgba_capture.png' (with alpha channel).
